# Distant Viewing Tutorial 2: Network-Era Sitcoms and Visual Style

[[**slides**](https://distantviewing.org/tutorial/dvt_tutorial_slides_02.pdf)]
[[**chapter**](https://direct.mit.edu/books/oa-monograph/chapter-pdf/2163344/c003200_9780262375160.pdf)]
[[**homepage**](https://www.distantviewing.org)]

This notebook explores the theory and methods
introduced in the book *Distant Viewing* (MIT Press, 2023) to study visual
style in two network era sitcoms. Specifically, we will look at every televised
episode of the series *Bewitched* (1964-1972) and *I Dream of Jeannie*
(1965-1970). In the notebook we will first walk through the methodology using a
short 60 second sample video from a later sitcom, *Friends* (1994-2004), and
going slowly through all of the steps. Then,
due to time, file size, and copyright constraints, we will load a precomputed
set of image annotations mirroring those from the 60 second sample and then
use this larger set for the purpose of analysis. Here are the specific learning
outcomes for the tutorial:

1. Explain how digital videos can be understood as a sequence of images.
2. Apply functions in the distant viewing toolkit to a short video file.
3. Calculate shot breaks using a state-of-the-art computer vision algorithm
and connect shot breaks to research questions in media studies.
4. Classify the estimated identity of characters using computer visional
algorithms and face embeddings.
5. Illustrate how to address humanities research questions using computer
vision algorithms applied to a corpus of television shows.

This notebook does not require any previous knowledge of Python or computer
vision. However, it moves fairly quickly through the preliminary steps of
working with digital images and only explains the most important aspects of
the Python code in each step. For a more in-depth introduction to how computers view images images  and python, we recommend first following the
 Distant Viewing Tutorial: Movie Posters and Color Analysis notebook using the movie posters corpus, which
can be accessed [here](https://colab.research.google.com/github/distant-viewing/dvt/blob/main/tutorials/Distant_Viewing_Tutorial_1_Movie_Posters_and_Color_Analysis.ipynb). For more about the Python package that we built, Distant Viewing Toolkit (**dvt**), please visit [our GitHub repository](https://github.com/distant-viewing/dvt).

## 2.1 Setup

As a first step, we need to install a few additional Python components,
download the datasets, and tell Python all of the functions
that we will need later in the tutorial. To get started, we will use the
code below to install the module called **dvt** (the distant viewing toolkit),
which contains several useful functions specifically designed to apply computer
vision algorithms to collections of humanities data. The exclamation point at
the start of the line of code tells the notebook that we we want to directly run a command line tool outside of Python itself. Here, we are using the tool
called **pip** that can used to install additional functionality for Python.
To run the code, hover your mouse somewhere over the background of the
code. This will show a triangular play button on the left-hand side of the code
block. Hit the button and wait for it to finish, which may take a minute or
two as Colab always takes a bit of time to set up when running the first
code block.

The computer vision models in this notebook run much faster on a graphics
card (GPU). The notebook asks Colab for one when it is opened. If you would
like to check, open the *Runtime* menu, choose *Change runtime type*, and
make sure that a GPU such as *T4 GPU* is selected before running any code.

In [ ]:
!pip install -q "dvt[video,face]"

Next, below we have some code to download the files that
we are working with in this notebook. These use the program **wget** to download
a short sample video and a set of character portraits from the toolkit's
GitHub repository, followed by three tables of metadata from the distant
viewing website. We will explore the
individual contents of each of these components as needed in the sections below.
The last line uses the **dvt** command itself to download the computer vision
models that are needed in the later sections of the notebook for finding shot
breaks and faces. As above, hover over the code and then click on the play button on
the left-hand side of the code block. This will run all of the lines of code
one after another. We should be able to see a small green arrow showing the
specific line being run at any given point, helping us understand which steps
take the most time to execute.

In [ ]:
!wget -q -nc "https://raw.githubusercontent.com/distant-viewing/dvt/main/tutorials/data/friends_sd_60s.mp4"
!wget -q -nc -P faces "https://raw.githubusercontent.com/distant-viewing/dvt/main/tutorials/data/friends_faces/chandler.jpg"
!wget -q -nc -P faces "https://raw.githubusercontent.com/distant-viewing/dvt/main/tutorials/data/friends_faces/janice.jpg"
!wget -q -nc -P faces "https://raw.githubusercontent.com/distant-viewing/dvt/main/tutorials/data/friends_faces/laundromat_woman.jpg"
!wget -q -nc -P faces "https://raw.githubusercontent.com/distant-viewing/dvt/main/tutorials/data/friends_faces/phoebe.jpg"
!wget -q -nc -P faces "https://raw.githubusercontent.com/distant-viewing/dvt/main/tutorials/data/friends_faces/rachel.jpg"
!wget -q -nc "https://distantviewing.org/tutorial/ttl_sitcom_metadata.csv"
!wget -q -nc "https://distantviewing.org/tutorial/ttl_sitcom_characters.csv"
!wget -q -nc "https://distantviewing.org/tutorial/ttl_sitcom_shots.csv"
!dvt models fetch --annotators frames,face

In the last part of the setup, we will run some code in Python (note that the
lines below do not start with an exclamation mark). This code here uses
the **import** command to tell Python which libraries and functions we are
going to use in the notebook. Specifically, these are **os** for working with
files on the operating system, **numpy** for working with large arrays of
numbers, **pandas** for working with tabular datasets,
**matplotlib.pyplot** for data visualization, **cv2** for reading images and
the frames of a video, two functions for displaying a video and an image inside
of the notebook, and the **dvt** module that we described above.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import dvt
from IPython.display import Video
from google.colab.patches import cv2_imshow

In Google Colab, your working environment resets itself everytime you reopen a
notebook. Therefore, all of the steps above need to be re-run each time that
you start the notebook. If you were running this code on your own machine, the
installation of the **dvt** package and downloading the data would only need to
be done once. Loading the modules in the final code chunk, however, always needs
to be run each time that Python is restarted.

## 2.2 Network-Era Sitcom Dataset

Before we start looking at the computational steps needed to work with moving
image data, it's helpful to understand the humanities research questions that
motivate this work. Here, we are concerned with two popular U.S. sitcoms from
the 1960s and early 1970s: *Bewitched* (1964-1972) and *I Dream of Jeannie*
(1965-1970). Here is metadata abotu each of the episodes from the entire runs
of the two shows:

In [ ]:
meta = pd.read_csv("ttl_sitcom_metadata.csv")
meta

These two series are often compared and constrasted to one another, with
*I Dream of Jeannie* being seen as an attempt by NBC to copy the success of
ABC's *Bewitched*, which started one season prior. While a lot has been written
about the cultural significance of these two shows, prior research had not
seriously considered the visual style of the two shows. We are interested in
how the way that the shows are shot and edited contribute to our understanding
of questions such as who is/are the main characters, to what extent does the
visual style contribute to relationships between the characters, and how do
these aspects change over time. The latter is particularly interesting because
both shows started their broadcasts in black-and-white before transitioning
to color for the majority of the later seasons.

## 2.3 Sample Movie File

Now that we have some ideas of the big questions we are interested in, let's
look at a short sample to understand the computational
steps needed to work with moving images. When doing computational analyses with
visual and audiovisual materials, it is
important to frequently go back and look at specific examples to
ensure that we understand how our large-scale analysis connects back to the
actual human experience of viewing. In this notebook we are going to start by
working slowly with the process of creating annotations that summarize a short
60 second clip from the fifth episode of the first season of the show
*Friends*. It is a sitcom from a later period than the two series above, but the
steps are exactly the same for any video file. Running the code below shows an
embedded version of the clip that can be watched within the Colab notebook.

In [ ]:
Video("friends_sd_60s.mp4", embed=True, width=640)



We suggest watching the clip a couple of times. Try to pay attention (and maybe
even write down) the number of shots in the clip and how they are framed. In
the next few sections, we will use computer vision algorithms to try to
capture these features as structured annotations.

## 2.4 Working with Video Frames as Images

Video file formats such as **mp4**, **avi**, and **ogg** typically store moving
images in a complex format that is highly optimized to reduce file sizes and
decompression time. When working with moving images computationally in programs
such as Python, however, we typically decompress these files and turn them
into a sequence of individual images stored as arrays of pixels, with one
image for each frame in the input video. The distant viewing toolkit  (**dvt**)
is built around a single function called `annotate`. We give it a video
file, tell it where to save its results, and name one or more *annotators*
that we want to run. Each annotator creates a table of structured data about
the video. In this section, we will demonstrate how this works.

First, we can use the `metadata` annotator to access metadata about a video
file. Here, we will load the metadata for our sample video file and display
the number of frames, the rate of playback given as frames per second (`fps`),
and the width and height of each frame in pixels.

In [ ]:
tables = dvt.annotate("friends_sd_60s.mp4", out="output", annotators=["metadata"])
info = tables["metadata"].to_pandas()[["n_frames", "fps", "img_width", "img_height"]]
info

As we see in the metadata, even this short clip has 1439 individual frames.
When working with an entire television episode, let along a feature-length
movie file, it quickly becomes difficult to load all of frames from a video
into Python at once. The image sizes, particularly once we start looking at
materials in high-definition, are just too large to hold these all in even a
computer's memory at the same time.


An alternative approach is to write
a loop which loads in each frame of the video file one by one. The **cv2**
library has a function called `VideoCapture` for doing this.
Specifically, each time its `read` method is called, it returns the next frame
as an array of pixels, along with a flag that becomes false when the end of the
file has been reached.

The code below show an example of how this works. We
cycle through each of the frames. For each frame we store the frame number,
the time in seconds since the start of the video, and the average pixel value
(a measurement of the frame's
brightness). Then, we turn this into a tabular data frame with one row for
each frame in the video.

In [ ]:
output = {'frame': [], 'time': [], 'brightness': []}
fps = info['fps'][0]

video = cv2.VideoCapture("friends_sd_60s.mp4")
frame = 0
while True:
  ok, img = video.read()
  if not ok:
    break
  output['frame'].append(frame)
  output['time'].append(frame / fps)
  output['brightness'].append(np.mean(img))
  frame += 1
video.release()

output = pd.DataFrame(output)
output

Brightness is one of the simpliest annotations that we can use to summarize an
image. But, even this simple measurement can already provide a rough way of
representing our video clip. To see this, let's plot the brightness of the
frame over time.

In [ ]:
plt.plot(output['time'], output['brightness'])
plt.show()

Looking at the brightness, particularly the large jumps in the values, can you
identify the shot breaks that you found when watching the video? Some of the
breaks are easy to see in the plot. Are there any that hardly show up at all?
What is happening in the video in those places?

## 2.5 Shot Boundary Detection

One of the first tasks that we often want to perform on a video file is to
identify the breaks between shots. This process is called *shot boundary
detection*. As we saw in the brightness plot above, some simple heuristics
can be used to approximately identify many kinds of shot breaks. If we want
more accurate predictions, which do not falsely identify quick motion as a
shot boundary or fail to find subtle breaks such as cross-fades, we need
to make use of a more complex algorithm. The distant viewing toolkit (**dvt**) includes
a neural-network based shot boundary detection algorithm that we have found
works well across many different genres and input types.

To run the built-in shot boundary detection algorithm, we call the `annotate`
function again and ask for two annotators. The first, `frames`, runs the
neural network over every frame of the video and records how likely it is that
a shot ends there. The second, `shots`, uses those numbers to cut the video
into shots. The Python
code will print out its progress through the file as it runs. Because we
are saving the results in the same place as before, the tables that we have
already created are returned again alongside the new ones.

In [ ]:
tables = dvt.annotate("friends_sd_60s.mp4", out="output", annotators=["frames", "shots"])

The `shots` table has one row for each detected shot and many columns that
describe it. We will convert it into a pandas data frame and keep only a few
of them: the number of the shot (`sid`), its first and last frames (`f0` and
`f1`), its start and end times in seconds (`t0` and `t1`), and its middle frame
(`fmid`). We will also add the length of each shot in seconds. Here is what
the algorithm found for this video clip:

In [ ]:
shot = tables["shots"].to_pandas()[["sid", "f0", "f1", "t0", "t1", "fmid"]]
shot['length'] = shot['t1'] - shot['t0']
shot

Take a few minutes to confirm that these shot boundaries correspond to the ones
that you found when watching the video. You can even rewatch (and pause at each
break) the video and see that it corresponds with the breaks that were
automatically identified in the shot boundary detection.

In the next section we will look at face detection and identification. This is a
common task when working with movie image data but ultimately works using
individual images. To make this a bit easier to work with at first, we will use
the following code to build a dataset with one frame from each of the detected
shots in the video clip file. Specifically, we grab the middle frame from each
shot, as specified in the `fmid` column of the `shot` data frame above.

In [ ]:
img_list = []
video = cv2.VideoCapture("friends_sd_60s.mp4")
for frame in shot['fmid']:
  video.set(cv2.CAP_PROP_POS_FRAMES, frame)
  ok, img = video.read()
  img_list.append(img)
video.release()

To get a sense of what these look like, we can convert the images
into thumbnails and combine them into a grid to get a view of what the shots
from our sample video file look like. The shots are shown in order, reading
across each row from left to right.

In [ ]:
thumbs = [cv2.resize(img, (img.shape[1] // 4, img.shape[0] // 4)) for img in img_list]
while len(thumbs) % 5 != 0:
  thumbs.append(np.zeros_like(thumbs[0]))

img_comb = np.vstack([np.hstack(thumbs[i:(i + 5)]) for i in range(0, len(thumbs), 5)])
cv2_imshow(img_comb)

We will use the full versions of these images in the next section as we
identify the location, size, and identity of the characters in each shot.

## 2.6 Face Detection and Facial Recognition

Locating and identifying the faces present in a particular frame of a video
file is a common way of understanding the narrative structure and visual style
of the source material. In this section, we will see how to work with faces
using the distant viewing toolkit (**dvt**), looking closely at the
images from the sample video file that we extracted in the previous section.
After seeing how this works with a single image, in the following
section we will see how to put this together in a way that would scale to
larger video files.

To get started, we will run the `face` annotator over the sample video. It
looks for faces in every frame of the video, which takes longer than
the steps above. With a GPU it should finish in well under a minute. The
result is a table with one row for each face found in each frame.

In [ ]:
tables = dvt.annotate("friends_sd_60s.mp4", out="output", annotators=["face"])
face = tables["face"].drop("embedding", "kps", "version").to_pandas()
face

Each row gives the frame in which a face was found, a score between 0 and 1
describing how confident the algorithm is that it has found a face
(`det_score`), and the location of a box around the face, in pixels. To see
what this looks like, let's take the middle frame of the eleventh shot
(remembering that Python starts counting at zero, so the
eleventh image is selected by selecting the image in position 10), select the
rows of the table for that frame, and draw each of the boxes onto a copy of
the image.

In [ ]:
here = face[face['frame'] == shot['fmid'][10]]

img = img_list[10].copy()
for box in here.itertuples():
  top_left = (int(box.bbox_x1), int(box.bbox_y1))
  bottom_right = (int(box.bbox_x2), int(box.bbox_y2))
  cv2.rectangle(img, top_left, bottom_right, (0, 255, 0), 2)

The object called `img` now has a copy of the frame along with a box
around any detected faces. Here we see that the algorithm has detected one face,
which is likely the number of faces a human annotator would also have found in
the frame.

In [ ]:
cv2_imshow(img)

The visualization of the detected face is great for a one-off understanding of
the algorithm. In order to use this information for
computational analyses, we need to have the location of the detected face
represented in a strutured format. This is exactly what the rows of the `face`
table provide. Here is
all of the structured information about the face detected in the frame above:

In [ ]:
here

If you followed along with the introductory notebook using movie posters, this
is the same information that we used to detect the number of faces present on
each poster. One element that makes working with moving images different is that
we usually have the same people showing up over and over again across frames and
shots. Labeling the identity of the people present in each shot is an essential
step in understanding the narrative structure of the material. The distant
viewing toolkit (**dvt**) also includes information for doing facial recognition. This
requires a bit more care as we need to start by identifying the characters that
we want to identify in the first place.

When we run the face detection algorithm, each detected faces is
also assocaited with a sequence of 512 numbers called an *embedding*. The
individual numbers do not have a direct meaning, but are instead defined
in a relative way such that if two images of the same person are each associated
with an embedding, we would expect the embeddings to be more similar to one
another than they are to the embeddings of other faces. The embeddings are
stored in a column of the `face` table that we left out above because it is
too large to display. In the code below we grab all of the embeddings as an
array of numbers, with one row for each detected face, and rescale each
embedding so that they can be easily compared with one another. Then, we
can see the shape of the array: one row for each face in the table and
512 columns.

In [ ]:
embed = tables["face"]["embedding"].to_numpy().astype(np.float32)
embed = embed / np.linalg.norm(embed, axis=1, keepdims=True)
embed.shape

One way to use these embeddings is to first identify portraits of the actors
that we are interested in from the video file, and then associate each of these
with a character name. Then, we can compute the embeddings of these faces and
store them. Finally, each time we detect a face in a frame, we can check how
similar the embedding of the detected face is compared to the faces in our
reference set. If one of the reference characters is sufficently close, we
will assume that we have found the associated character.

One of the objects that we downloaded in the setup section of this notebook was
a folder with portraits of five characters who appear in our sample, taken
from elsewhere in the same episode of *Friends* and named with
the name of the character in the show. The `annotate` function can be given
a folder of images in place of a video file, and returns the same kind of
table with the name of each image file in place of a frame number. In the
code below we run the `face` annotator over the portraits and record the name
and embedding of each of the faces. Also,
for reference, we combine the portraits into a single image.

In [ ]:
portraits = dvt.annotate("faces", annotators=["face"])["face"]

face_name = np.array([path[:-4] for path in portraits["path"]])
face_embed = portraits["embedding"].to_numpy().astype(np.float32)
face_embed = face_embed / np.linalg.norm(face_embed, axis=1, keepdims=True)
face_img = np.hstack([cv2.imread("faces/" + path) for path in portraits["path"]])

Here are the portraits of the characters. They are ordered in alphabetical
order: Chandler, Janice, a woman at the laundromat, Phoebe, and Rachel.

In [ ]:
cv2_imshow(face_img)

In order to measure the similarity of two embeddings, we use a mathematical
technique called a *dot product*. This number will be `1` if two embeddings
are exactly the same and `-1` if they are exactly opposite one another.
Typically, two faces that are of different people will have a similar score
somewhere close to zero. Let's start by seeing the similarity scores between
the set of five portraits.

In [ ]:
np.dot(face_embed, face_embed.T).round(2)

The first column is Chandler, and compares Chandler to each of the five
images. Since he is the first image, we see `1` in the first row because the
photo is being compared to itself. The rows below it compare the image of
Chandler to Janice, the woman at the laundromat, Phoebe, and Rachel.

The values along the diagonal are all 1 (or very close to one) because we are
comparing one embedding to itself. All of the other similarity scores are
close to zero, with none of them larger than `0.1`, which is as expected since
each of the portraits shows a unique actor.

Now, let's compare these reference faces to the embedding of the face in the
individual frame that we started this section with: the middle frame from the
eleventh shot (`img_list[10]`).

In [ ]:
dist = np.dot(face_embed, embed[here.index].T)
dist

Here we see that there is a much larger similarity score (around `0.6`)
between this face and the third character in our set. And if we look at the
image, we see in fact that this is the same character: the woman at the
laundromat.

 We can associate the face with the
character name algorithmically using the following code.

In [ ]:
face_name[np.argmax(dist, axis = 0)]

The code above associates the face with the character that has the highest
similarity. It would be a good idea to also store the similarity score
and then only trust the relationship if this score is sufficently large, which we can do with the following code.



In [ ]:
np.max(dist, axis = 0)

Now that we understand how this process works for a single image, let's apply
this technique to every face in the sample file.

## 2.7 Building Annotations from a Video File

Because the `face` annotator has already looked at every frame of the video,
we can compare the reference portraits to all of the detected faces at once.
In the code below we find the most similar character for each row of the
`face` table and store the name and the similarity score as two new columns.
To start, we will look only at the faces in the middle
frame of each detected shot, keeping the faces that the algorithm is fairly
confident about.

In [ ]:
dist = np.dot(face_embed, embed.T)
face['character'] = face_name[np.argmax(dist, axis = 0)]
face['confidence'] = np.max(dist, axis = 0)

output = face[face['frame'].isin(shot['fmid'])]
output = output[output['det_score'] > 0.7]
output

Take a few minutes to go back to the thumbnail images from each of these
detected faces and see how they line up with the characters that are actually
present (if you are not familiar with *Friends*, use the portraits to learn
each of the character names). You should see that the faces with a high
confidence score line up well. Notice, though, that every face has been given a
name, including the people in the background of the coffee shop and the
laundromat who are not in our set of portraits at all. Their confidence scores
are close to zero. This is why we need to store the similarity score and only
trust a name when the score is sufficiently large. We always recommend taking a look and reviewing the results as you go. Then, we can adjust our approach as needed based on the audiovisual data that we are working with and the areas of inquiry animating our analysis.

Having the faces from every frame, rather than only one frame from each shot,
has some advantages. For example, we can measure how long each character is
visible. In the code below we keep only the faces with a similarity score
above `0.4`, count the number of frames in which each character appears, and
divide by the number of frames per second to convert the result into seconds.

In [ ]:
found = face[face['confidence'] > 0.4]
found.groupby('character')['frame'].nunique() / fps

In this one minute sample, Chandler and Rachel are each on screen for the
longest amount of time. We will return to this kind of measurement, the amount
of time that a character is visible, at the end of the next section.

## 2.8 Full Corpus Analysis

We now have all of the methods and code needed to identify shots and faces in a
moving image file. If we had an entire episode of *Friends*, we could run the
exact same sequence of steps above on the full episode without changing
anything. The only difference would be that the results would take much longer
to finish running. If we had a folder with every episode of the series, we
would just need to add a loop to cycle over each video
file and make sure to include information about the filename on each row of
the output. Finally, if we wanted to extend this to another series, we would
just need to add the portraits of any characters that we wish to identify to
our folder of reference images.

We now return to the two shows, *Bewitched* and *I Dream of Jeanine*, that are central to Chapter 5 of *Distant Viewing*. The chapter centers around a comparison of the two magical sit-coms, which we can do by putting together shot boundary and then face detection and recognition.   

The full set of video files for the two shows are quite large and the files are under copyright.
It also takes a long time to process all of these files, particularly within a
free Colab session. Since it is not possible to run the annotations directly
on the full set here, we will instead work with the pre-computed annotations
that were downloaded during the notebook setup. The structure of the dataset
includes one row for each shot, with data about the timing of the shot and
the number of faces.

In [ ]:
shots = pd.read_csv("ttl_sitcom_shots.csv")
shots

One quick metric that we can compute is the average shot length for each of the two
series. This is a commonly used measurement to understand the pacing of a movie
or television show. Here we see that Bewitched is slightly faster, with an
average shot length of 5.3 seconds compared to the 6.2 seconds of *I Dream of
Jeannie*.

In [ ]:
shots.groupby(['series'])['time'].mean().sort_values()

A related measurement is the median shot length, which here shows that the
median shot length is slightly longer in *Bewitched* compared to
*Jeannie*. So, while a shot in *Bewitched* is slightly longer, there may
be a set of particularly long shows in *Jeannie* that causes the average
of that show to be longer.



In [ ]:
shots.groupby(['series'])['time'].median().sort_values()

One of the interesting patterns that we noticed when writting the chapter based
on these two series is that the average shot length is closely related to the
number of faces present on the screen. We decided to truncate the number of faces to
three (so that anything greater than 3 faces becomes 3) given our knowledge of the two shows. Notice how the
average shot length increases with the number of faces on screen in both
series.

In [ ]:
shots.loc[shots['num_face'] > 3, 'num_face'] = 3
shots.groupby(['series', 'num_face'])['time'].mean()

This
helps validate that the shot length is related to visual style. At least for
these two series, it is related to how often we see a close up on an individual
character talking (or acting) versus seeing multiple characters interacting
together in a longer shot.

We also have a dataset that indicates the specific characters detected in shots
from the two series. This uses a face detection algorithm similar to the one
that we saw used in our sample video file. Here, we have one row for each
detected character in a shot. There are four main characters in each of the
series.

In [ ]:
characters = pd.read_csv("ttl_sitcom_characters.csv")
characters

We can use the average amount of time that a character is on screen in a given
episode as a rough measurement of the character's visual importance in the
show. Here, we compute this metric for each of the characters and sort in
descending order within the series.

In [ ]:
temp = characters.groupby(['series', 'video', 'character'])['time'].agg('sum').reset_index()
temp = temp.groupby(['series', 'character'])['time'].agg('mean').reset_index()
temp['time'] = temp['time'] / 60
temp.sort_values(['series', 'time'], ascending=False)

And, already we see one of the main conclusions of the analysis of these series:
while they are thought of as very similar, their narrative structures are
quite different. *Jeannie* is actually centered on the male lead Tony; the
titual character Jeannie is often little more than a plot device who sets the
action in motion. In constrast, *Bewitched* is most focused on the relationship
between Samantha and Darrin, which each of them having nearly the same amount
of overall screen time.

Our annotations - shot detection, face detection, and face recognition - can now be analyzed from many different angles to explore different aspects of the shows, which we delve further into in Chapter 5. Key here is that just three annotations open up analytical possibilities, and can address humanities questions about audiovisual data.

## 2.9 Conclusions and Next Steps

This notebook has given an introduction to use the distant viewing toolkit (**dvt**) to
annotate movie image data from a digital video file. We've seen how to get
basic metadata about a video file from within Python, how to cycle through
the individual frames of a video file, and how to detect shot breaks using a
neural network. Although not specific to moving images, we also covered the
process of using computer vision algorithms to do facial recognition, which is
a common task in many applications but particularly useful when working with
film and television corpora. Finally, we loaded a larger dataset showing all
of the detected shots and characters across the entire runs of two U.S.
Network-Era sitcoms and performed several example analyses on them.

For readers interested in more details about the specific case study of these
two sitcoms, we suggest reading the fifth chapter of the
[*Distant Viewing*](https://www.distantviewing.org/book/) book, available
under an open access license. The first two chapters of the book may also be
of interest as they offer a more general theoretical and methodological
approach to the computational analysis of digital images.